In [1]:
import pandas as pd
import plotly.graph_objects as go

In [11]:
def generate_sankey_chart(csv_file, title="Trips in GIPUZKOA"):
    """Generates a Sankey chart from a CSV file with an optional title"""
    try:
        df = pd.read_csv(csv_file)

        # 👇 FILTER OUT same-prefix flows
        def get_prefix(name):
            return name.split('_')[0] if isinstance(name, str) and '_' in name else name

        if 'name_origen' in df.columns:
            df = df[
                df.apply(
                    lambda row: get_prefix(row['name_origen']) != get_prefix(row['name_destino']),
                    axis=1
                )
            ]

        if 'name_origen' in df.columns:
            # Create node labels
            labels = list(pd.unique(df[['name_origen', 'name_destino']].values.ravel('K')))
            
            # Map nodes to indices
            source_indices = [labels.index(src) for src in df['name_origen']]
            target_indices = [labels.index(dst) for dst in df['name_destino']]
        else:
            # Create node labels
            labels = list(pd.unique(df[['COMARCA_origen', 'COMARCA_destino']].values.ravel('K')))
            
            # Map nodes to indices
            source_indices = [labels.index(src) for src in df['COMARCA_origen']]
            target_indices = [labels.index(dst) for dst in df['COMARCA_destino']]

        # Assign all source nodes to x = 0.1 and destination nodes to x = 0.6
        x_positions = []
        for label in labels:
            if label in df['name_origen'].values:
                x_positions.append(0.1)
            elif label in df['name_destino'].values:
                x_positions.append(0.6)
            else:
                x_positions.append(0.3)  # default center if unclear
        
        # Create Sankey diagram
        fig = go.Figure(go.Sankey(
            node=dict(
                pad=15,
                thickness=20,
                line=dict(color="black", width=0.5),
                label=labels,
                x=x_positions
            ),
            link=dict(
                source=source_indices,
                target=target_indices,
                value=df['viajes'],
                color='rgba(150,150,150,1)'
            )
        ))

        # Set the title for the Sankey chart
        fig.update_layout(
            width=1000,
            height=1000,
            title=dict(
                text=title,
                x=0.5,  # Center title horizontally
                # y=1.2,  # Move title higher above the graph
                font=dict(size=40, color="black", family="Arial"),
                xanchor="center",
                yanchor="top"
            ),
            font=dict(color="white", size=25),  # Adjust text color for readability
            hoverlabel=dict(font=dict(color="black")),  # Keep hover text readable
            paper_bgcolor='rgba(0,0,0,0)',  # Fully transparent background
            plot_bgcolor='rgba(0,0,0,1)',  # Fully transparent plot area
            margin=dict(l=10, r=10, t=80, b=10)  # Reduce margins for better overlay
        )
        
        return fig

    except Exception as e:
        return {"data": [], "layout": {"title": f"Error loading CSV: {str(e)}"}}
    
fig = generate_sankey_chart("C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/regional_mobility/sankey_graphs/tolosaldea_sankey.csv", "Trips in TOLOSALDEA")
fig.show()